# Dependencies

In [16]:
# pandas and NumPy
import pandas as pd
import numpy as np

# Seaborn and matplotlib for visualization
import seaborn as sns
import matplotlib.pyplot as plt

# Classification models
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LinearRegression  # for linear models
from sklearn.naive_bayes import GaussianNB
from sklearn import svm

# K-Means clustering
from sklearn.cluster import KMeans

# Train / test split
from sklearn.model_selection import train_test_split

# Metrics
from sklearn import metrics


# Data pre-processing


In [17]:
df = pd.read_csv('laptops_data.csv')

df.head()

,Unnamed: 0,Name,Discounted Price,Actual Price,Saving,Rating,Reviews,Brand,Core,SSD,Model
0,0,Apple MacBook Air 13 M1 MGN63 (8GB-256GB),203499.0,258000.0,21% OFF,5.0,2.0,Apple,M1,NaN,MacBook Air 13 M1 MGN63
1,1,Apple Macbook Air 13 MW123 M4 Chip,281999.0,350000.0,19% OFF,4.3,5.0,Apple,M4,NaN,Macbook Air 13 MW123 M4 Chip
2,2,ASUS Zenbook 14 UX3405CA Intel Core Ultra 7 25...,285999.0,330000.0,13% OFF,NaN,NaN,ASUS,Ultra 7,16GB-512GB SSD,Zenbook 14 UX3405CA Intel Core Ultra 7 255H
3,3,Lenovo ThinkPad E16 Gen 2 - Intel Core Ultra 7...,287999.0,340000.0,15% OFF,NaN,NaN,Lenovo,Ultra 7,NaN,ThinkPad E16 Gen 2 - Intel Core Ultra 7
4,4,Lenovo IdeaPad Slim 3 Ryzen 7 (8GB-512GB),146999.0,175000.0,16% OFF,NaN,NaN,Lenovo,Ryzen 7,NaN,IdeaPad Slim 3 Ryzen 7


In [18]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 306 entries, 0 to 305
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Unnamed: 0        306 non-null    int64  
 1   Name              306 non-null    object 
 2   Discounted Price  306 non-null    float64
 3   Actual Price      291 non-null    float64
 4   Saving            291 non-null    object 
 5   Rating            26 non-null     float64
 6   Reviews           26 non-null     float64
 7   Brand             306 non-null    object 
 8   Core              219 non-null    object 
 9   SSD               92 non-null     object 
 10  Model             306 non-null    object 
dtypes: float64(4), int64(1), object(6)
memory usage: 26.4+ KB
None


In [19]:
df.isnull().mean() * 100


Unnamed: 0           0.000000
Name                 0.000000
Discounted Price     0.000000
Actual Price         4.901961
Saving               4.901961
Rating              91.503268
Reviews             91.503268
Brand                0.000000
Core                28.431373
SSD                 69.934641
Model                0.000000
dtype: float64

- Rating, reviews, and SSD contain a high proportion of missing values and cannot be reliably imputed, so these columns are removed.

The first column is unnamed (it corresponds to an automatically generated index from the CSV file), so it is also dropped.

The Name column contains multiple pieces of information. Since the model name is already available in the Model column, 
we extract the RAM and storage information from Name into two separate columns (RAM and Storage).
After this extraction, the Name column becomes redundant and is therefore removed.

In [28]:
df1 = df.iloc[:, 1:]

# Extract RAM
df1['RAM_GB'] = df1['Name'].str.extract(r'(\d+GB)', expand=False)

# Extract Storage
df1['Storage_GB'] = df1['Name'].str.extract(r'-(\d+GB)', expand=False)

# Clean and convert
df1['RAM_GB'] = df1['RAM_GB'].str.replace('GB', '', regex=False).astype('float')
df1['Storage_GB'] = df1['Storage_GB'].str.replace('GB', '', regex=False).astype('float')

# Drop Name, Rating, Reviews, and SSD
df1 = df1.drop(columns=['Name', 'Rating', 'Reviews', 'SSD'])

df1.head()


,Discounted Price,Actual Price,Saving,Brand,Core,Model,RAM_GB,Storage_GB
0,203499.0,258000.0,21% OFF,Apple,M1,MacBook Air 13 M1 MGN63,8.0,256.0
1,281999.0,350000.0,19% OFF,Apple,M4,Macbook Air 13 MW123 M4 Chip,NaN,NaN
2,285999.0,330000.0,13% OFF,ASUS,Ultra 7,Zenbook 14 UX3405CA Intel Core Ultra 7 255H,16.0,512.0
3,287999.0,340000.0,15% OFF,Lenovo,Ultra 7,ThinkPad E16 Gen 2 - Intel Core Ultra 7,16.0,512.0
4,146999.0,175000.0,16% OFF,Lenovo,Ryzen 7,IdeaPad Slim 3 Ryzen 7,8.0,512.0


## Correlation matrix